# 05 保存済み追跡実験の群れ評価

run_experiments.pyで保存した全条件の結果を読み込み，群れ中心，非観測時間，予測成功率を比較する．実験実行と中断再開はNotebookでは行わない．

## 1．分析対象を設定する

run_experiments.pyで使用したCase IDとBenchmark IDを指定する．

In [ ]:
CASE_ID = "MiyotaMtBase_gsi_3km_v3"  # 使用する農場Case
BENCHMARK_ID = "miyota-group-tracking-v1"  # 条件別結果をまとめるID
DATASET_IDS = None  # Noneなら全Dataset，または["dataset-id"]
LAYOUT_IDS = None  # Noneなら全配置，または["layout-id"]


## 2．実行状態を確認する

実験設定やTrackerはここでは変更しない．Runnerが保存した実行Manifestを読み取る．

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from field_environment import find_repository_root

REPOSITORY_ROOT = find_repository_root()
CASES_ROOT = Path(os.environ.get("WILDLIFE_CASES_ROOT", REPOSITORY_ROOT / "cases"))
benchmark_directory = CASES_ROOT / CASE_ID / "benchmarks" / BENCHMARK_ID
execution_manifest = json.loads(
    (benchmark_directory / "execution_manifest.json").read_text(encoding="utf-8")
)
status_table = pd.DataFrame(execution_manifest["conditions"].values())
status_table

## 3．完了済み結果を選択する

In [ ]:
if not (status_table["status"] == "COMPLETED").any():
    raise ValueError("完了済みの実験条件がない．")
status_table[status_table["status"] == "COMPLETED"]

## 4．保存済み結果を読み込む

結果が存在しない場合は，先にworkspace/run_experiments.pyを実行する．

In [ ]:
benchmark_directory = CASES_ROOT / CASE_ID / "benchmarks" / BENCHMARK_ID
results = pd.read_parquet(benchmark_directory / "condition_results.parquet")
if DATASET_IDS is not None:
    results = results[results["dataset_id"].isin(DATASET_IDS)]
if LAYOUT_IDS is not None:
    results = results[results["layout_id"].isin(LAYOUT_IDS)]
print(f"読込元: {benchmark_directory}")
print(f"実行Run数: {len(results):,}")

## 5．群れ予測結果を比較する

主結果は10 m以内の予測成功率である．5 m・20 mは閾値に対する感度確認として併記する．

In [ ]:
display_columns = [
    "dataset_id", "layout_id", "tracker_method", "visibility_coverage_rate",
    "hidden_state_count", "mean_hidden_position_error_m",
    "prediction_success_rate_5m", "prediction_success_rate_10m",
    "prediction_success_rate_20m", "reacquisition_success_rate",
]
results[display_columns].sort_values(
    ["dataset_id", "layout_id", "tracker_method"]
).reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(10, 7))
for (layout_id, tracker_method), group in results.groupby(["layout_id", "tracker_method"]):
    axis.scatter(
        group["visibility_coverage_rate"],
        group["prediction_success_rate_10m"],
        s=90,
        label=f"{layout_id} / {tracker_method}",
    )
axis.set_xlabel("2D visibility coverage rate")
axis.set_ylabel("Prediction success rate within 10 m")
axis.set_xlim(0.0, 1.0)
axis.set_ylim(0.0, 1.0)
axis.grid(True, alpha=0.3)
axis.legend(fontsize=8)
plt.show()

## 6．非観測時間別の結果を確認する

In [ ]:
duration_tables = []
case_directory = CASES_ROOT / CASE_ID
for row in results.itertuples(index=False):
    path = case_directory / "runs" / row.run_id / "group_duration_summary.parquet"
    table = pd.read_parquet(path)
    if table.empty:
        continue
    table.insert(0, "tracker_method", row.tracker_method)
    table.insert(0, "layout_id", row.layout_id)
    table.insert(0, "dataset_id", row.dataset_id)
    duration_tables.append(table)
duration_results = pd.concat(duration_tables, ignore_index=True) if duration_tables else pd.DataFrame()
duration_results

## 7．Rerun 2D記録について

Rerun Web Viewerまたは保存済み`recording.rrd`で確認するRunを結果表から選ぶ．Notebookから表示処理は起動しない．

In [ ]:
results[["run_id", "dataset_id", "layout_id", "tracker_method"]]

## 8．評価式と新しいTrackerの追加

- 評価式と妥当性：packages/field-benchmark/README.md
- Tracker追加方法：workspace/tracking/README.md
- 新しいTrackerはworkspace/tracking/<方式名>.pyへcreate_tracker()を実装し，Runnerの`--tracker-method`へ指定する．